# t-SVGP

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import abc
from functools import partial
from typing import Dict, Optional, Tuple
from jaxtyping import ArrayLike, Bool, Float, Int, PyTree
import numpy as np
import optax 

import jax
import jax.numpy as jnp
from jax import Array
import chex
import equinox as eqx
import jax.scipy as jsp

from jax_mc_pilco.model_learning.gp.kernels.base import Kernel
from jax_mc_pilco.model_learning.gp.means import ZeroMean, Mean

In [3]:
from jax_mc_pilco.model_learning.tsvgp.src.models.tsvgp import t_SVGP
from jax_mc_pilco.model_learning.gp.kernels.stationary import ExpSquared

In [4]:
def softplus(X: ArrayLike) -> jax.Array:
    return jnp.log(1 + jnp.exp(X))

In [33]:
def natgrad_step(params: Dict, sites: Dict, model: eqx.Module, *, lr=0.1) -> Tuple[Dict, Dict]:
    """Takes natural gradient step in Variational parameters in the local parameters
    λₜ = rₜ▽[Var_exp] + (1-rₜ)λₜ₋₁
    Input:
    :param: X : N x D
    :param: Y:  N x 1
    :param: lr: Scalar

    Output:
    Updates the sites
    """

    def compute_ve(mean: ArrayLike, var: ArrayLike, params: Dict, model: eqx.Module) -> Array:
        return model.variational_expectations(mean, var, params)

    kernel = model.kernel(**params["kernel"])
    z = params['inducing_point_locations']

    mu, Sigma = model.predict(model.X, params, sites)

    value_and_grads_fn = jax.value_and_grad(compute_ve, argnums=(0, 1))

    # Call the new function
    ve, grads = value_and_grads_fn(mu, jnp.diagonal(Sigma), params, model)
    
    # Compute the projection matrix A from prior information
    K_uu = kernel(z, z) + model.jitter(model.num_inducing_points)
    K_uf = kernel(z, model.X) 
    chol_Kuu = jnp.linalg.cholesky(K_uu)
    A = jsp.linalg.cho_solve((chol_Kuu, True), K_uf).T

    # Then ret_grads needs to be beta*m+alpha, beta
    ret_grads = [
        jnp.einsum("nm,n->m", A, grads[0]),
        jnp.einsum("nm,no,n->mo", A, A, jnp.clip(grads[1],min=1e-8)),
    ]
    meanZ, _ = tsvgp.predict(z)
    # chain rule at f
    grad_mu = ret_grads[0] - 2.0 * jnp.einsum("mo,o->m", ret_grads[1], jnp.squeeze(meanZ)), ret_grads[1]

    lambda_2 = sites['lambda_2']
    lambda_1 = sites['lambda_1']
    # compute update in natural form
    lambda_1 = (1 - lr) * lambda_1 - lr * grad_mu[0][:,jnp.newaxis]
    lambda_2 = (1 - lr) * lambda_2 - lr * grad_mu[1]

    return params, {'lambda_1': lambda_1, 'lambda_2': lambda_2}

In [6]:
def tsvgp_fit(
    model: t_SVGP,
    *,
    max_iters: Int = 500,
    n_e_steps: Int = 8,
    n_m_steps: Int = 20,
    nat_lr: Float = 0.8,
    adam_lr = 0.1,
) -> t_SVGP:
    """ Maximize the model ELBO using the dual parameterization method."""
    sites = model.sites
    params = model.params

    @jax.jit
    def params_loss(params: Dict, sites: Dict) -> Float:
        return -model.elbo(params, sites)

    # Initialise optimiser
    optimizer = optax.adam(learning_rate=adam_lr)

    opt_state = optimizer.init(params)
    params_loss_value_and_grad = eqx.filter_value_and_grad(params_loss)

    # Natural Gradient Step
    def natgrad_step_carry(index: Int, carry: Tuple[Dict,Dict])->Tuple[Dict,Dict]:
        params, sites = carry
        params, sites = natgrad_step(params,sites,model,lr=nat_lr)
        return (params,sites)

    # Optimization step.
    def optimization_step(index: Int, carry: Tuple[Dict, Dict, PyTree]) -> Tuple[Dict, Dict, PyTree]:
        """Need to run `n_e_steps` of the """
        params, sites, opt_state = carry
        loss_val, loss_gradient = params_loss_value_and_grad(params,sites)
        updates, opt_state = optimizer.update(loss_gradient, opt_state)
        params = optax.apply_updates(params, updates)

        return (params, sites, opt_state)

    # Optimisation loop
    for step in range(max_iters):
        params, sites = jax.lax.fori_loop(0,n_e_steps,natgrad_step_carry,(params,sites))
        params, sites, opt_state = jax.lax.fori_loop(0,n_m_steps,optimization_step,(params,sites,opt_state))
    
    return params, sites

## Parameters

In [7]:
n_e_steps = 8
n_m_steps = 20
nat_lr = 0.8
adam_lr = 0.1
M = 50
nm = 1
nit = 20
t_nit = n_e_steps * nit + n_m_steps * nit

mb_size = "full"
n_folds = 5

In [8]:
noise = 0.2

def func(X: ArrayLike)->Array:
    return np.sin(X)

# Noisy training data
X = np.arange(-3, 4, 1).reshape(-1, 1)
y = np.sin(X) + noise * np.random.randn(*X.shape)

X_test = np.arange(-5, 5, 0.2).reshape(-1, 1)
f_true = func(X_test)

In [9]:
num_inducing_points = 25
initial_inducing_points = jnp.linspace(X.min(),X.max(),num_inducing_points,endpoint=True)
params = {"kernel": {"coefficient": jnp.array(0.9),"log_scale": jnp.array(0.0),},"mean": {},"likelihood": {"log_diag": jnp.array(-0.5),},
         'inducing_point_locations': initial_inducing_points}

In [19]:
sites = {'lambda_1': jnp.zeros((num_inducing_points, 1)),'lambda_2': jnp.eye(num_inducing_points) * 1e-3}

In [20]:
tsvgp = t_SVGP(ExpSquared,X,y,params,sites)

In [21]:
tsvgp.elbo(params,sites)

/workspace/src/jax_mc_pilco/model_learning/tsvgp/src/models/tsvgp.py:362: UserWarning: You are calling predict on an unoptimized gp.
  warnings.warn("You are calling predict on an unoptimized gp.")


Array(-3316199.64833406, dtype=float64)

In [22]:
#opt_params, opt_sites = tsvgp_fit(tsvgp)

In [23]:
#tsvgp_opt = t_SVGP(ExpSquared,X,y,opt_params,opt_sites)

In [24]:
#mu, Sig = tsvgp_opt.predict(X_test)

In [25]:
#import matplotlib.pyplot as plt

In [26]:
# plt.scatter(X,y)
# plt.plot(X_test,f_true,color='black');
# plt.plot(X_test,mu,color='firebrick');

In [38]:
@jax.jit
def params_loss(params: Dict, sites: Dict) -> Float:
    return -tsvgp.elbo(params, sites)

# Initialise optimiser
optimizer = optax.adam(learning_rate=adam_lr)

opt_state = optimizer.init(params)
params_loss_value_and_grad = eqx.filter_value_and_grad(params_loss)

# Natural Gradient Step
def natgrad_step_carry(index: Int, carry: Tuple[Dict,Dict])->Tuple[Dict,Dict]:
    params, sites = carry
    params, sites = natgrad_step(params,sites,tsvgp,lr=nat_lr)
    return (params,sites)

# Optimization step.
def optimization_step(index: Int, carry: Tuple[Dict, Dict, PyTree]) -> Tuple[Dict, Dict, PyTree]:
    """Need to run `n_e_steps` of the """
    params, sites, opt_state = carry
    loss_val, loss_gradient = params_loss_value_and_grad(params,sites)
    updates, opt_state = optimizer.update(loss_gradient, opt_state)
    params = optax.apply_updates(params, updates)

    return (params, sites, opt_state)

In [39]:
p,s = jax.lax.fori_loop(0,n_m_steps,natgrad_step_carry,(params,sites))

/workspace/src/jax_mc_pilco/model_learning/tsvgp/src/models/tsvgp.py:362: UserWarning: You are calling predict on an unoptimized gp.
  warnings.warn("You are calling predict on an unoptimized gp.")


In [40]:
s

{'lambda_1': Array([[nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan],
        [nan]], dtype=float64),
 'lambda_2': Array([[-9.97706165e-09, -4.66172827e-11,  9.16727219e-11,
         -1.03835743e-10,  1.50534826e-10,  8.23013702e-12,
         -4.87794083e-11,  3.75548616e-11, -6.00843568e-12,
         -2.53203225e-11,  2.04633140e-11, -1.42345921e-12,
         -2.11961409e-11,  5.28288829e-12,  3.63935470e-12,
         -6.33463424e-12,  6.05235771e-12,  3.85574528e-12,
         -4.83933015e-12,  1.79680221e-12,  2.19355280e-12,
         -2.48765104e-12,  1.68980417e-12, -6.26774134e-13,
          2.32816662e-13],
        [-4.66172827e-11, -2.77613043e-11,  7.80693566e-11,
         -1.39427

In [77]:
pp,ss,os = jax.lax.fori_loop(0,n_m_steps,optimization_step,(params,s,opt_state))

In [80]:
# Optimisation loop
for step in range(10):
    params, sites = jax.lax.fori_loop(0,n_e_steps,natgrad_step_carry,(params,sites))
    params, sites, opt_state = jax.lax.fori_loop(0,n_m_steps,optimization_step,(params,sites,opt_state))

/workspace/src/jax_mc_pilco/model_learning/tsvgp/src/models/tsvgp.py:364: UserWarning: You are calling predict on an unoptimized gp.
  warnings.warn("You are calling predict on an unoptimized gp.")


## let's try to optimize this

In [25]:
# Define parameters
n_e_steps = 8
n_m_steps = 20
nat_lr = 0.8
adam_lr = 0.1
M = 50
nm = 1
nit = 20
t_nit = n_e_steps * nit + n_m_steps * nit

mb_size = "full"
n_folds = 5

data_name = "airfoil"  # Script can run:'boston', 'concrete', 'airfoil'
optim = "Adam"

In [12]:
from util import data_load
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

In [13]:
ell = 1.0
var = 1.0

data, test = data_load('airfoil', split=1.0, normalize=False)
X, y = data

In [14]:
X_scaler = StandardScaler().fit(X)
Y_scaler = StandardScaler().fit(y)
X = X_scaler.transform(X)
y = Y_scaler.transform(y)
N = X.shape[0]
D = X.shape[1]

# Initialize inducing locations to the first M inputs in the dataset
# kmeans = KMeans(n_clusters=M, random_state=0).fit(X)
# Z = kmeans.cluster_centers_
Z = X[:M, :].copy()

In [15]:
kf = KFold(n_splits=n_folds, random_state=0, shuffle=True)

RMSE = np.zeros((nm, n_folds))
ERRP = np.zeros((nm, n_folds))
NLPD = np.zeros((nm, n_folds))
TIME = np.zeros((nm, n_folds))

NLPD_i = np.zeros((nm, nit, n_folds))
LOGF_i = np.zeros((nm, nit, n_folds))

In [16]:
fold = 0
for train_index, test_index in kf.split(X):

    # The data split
    x = X[train_index]
    y = y[train_index]
    xt = X[test_index]
    yt = y[test_index]

    if mb_size == "full":
        mb_size = x.shape[0]

    train_dataset = (
        tf.data.Dataset.from_tensor_slices((x, y)).repeat().shuffle(x.shape[0])
    )

    mods, names = init_model(x.shape[0])

    maxiter = ci_niter(nit)

    j = 0

    for m in mods:
        t0 = time.time()
        logf_i, nlpd_i = run_optim(m, maxiter)
        t = time.time() - t0

        nlpd = -jnp.reduce_mean(m.elbo((xt, yt))).numpy()
        Eft, _ = m.predict_f(xt)
        rmse = jnp.sqrt(jnp.mean(jnp.square(yt - Eft)))

        yp, _ = m.predict_y(xt)
        errp = 1.0 - np.sum((yp > 0.5) == (yt > 0.5)) / yt.shape[0]

        print("NLPD for {}: {}".format(m.name, nlpd))
        print("ERR% for {}: {}".format(m.name, rmse))

        # Store results
        ERRP[j, fold] = rmse
        NLPD[j, fold] = nlpd
        TIME[j, fold] = t

        NLPD_i[j, :, fold] = np.array(nlpd_i)
        LOGF_i[j, :, fold] = np.array(logf_i)

        j += 1

    fold += 1

IndexError: index 1205 is out of bounds for axis 0 with size 1202

In [ ]:
# Calculate averages and standard deviations
rmse_mean = np.mean(ERRP, 1)
rmse_std = np.std(ERRP, 1)
nlpd_mean = np.mean(NLPD, 1)
nlpd_std = np.std(NLPD, 1)
time_mean = np.mean(TIME, 1)
time_std = np.std(TIME, 1)

elbo_mean = np.mean(LOGF_i, 2)
nlpd_i_mean = np.mean(NLPD_i, 2)

In [17]:
plt.title("ELBO" + "_" + data_name)
plt.plot(range(nit), elbo_mean[0, :][:], label=names[0])
plt.plot(range(nit), elbo_mean[1, :][:], label=names[1])
plt.plot(range(nit), elbo_mean[2, :][:], label=names[2])
plt.legend()
plt.show()

plt.title("NLPD" + "_" + data_name)
plt.plot(range(nit), nlpd_i_mean[0, :][:], label=names[0])
plt.plot(range(nit), nlpd_i_mean[1, :][:], label=names[1])
plt.plot(range(nit), nlpd_i_mean[2, :][:], label=names[2])
plt.legend()
plt.show()

NameError: name 'plt' is not defined